# Moliterate quick overview
This short tutorial shows how to load datasets, inspect a single entry,
iterate through entries, and perform simple dataset operations such as
concatenation, filtering, and slicing.

In [1]:
from pathlib import Path

import scm.moliterate as moliterate

moliterate_path = Path(moliterate.__path__[0])
TUTORIAL_FOLDER = moliterate_path.parent.parent.parent / "tutorials"

## Load your data
`load_dataset` accepts a path to a dataset (directory or file) and returns
an iterable dataset object that supports indexing and slicing.

In [2]:
from scm.moliterate import load_dataset

path = TUTORIAL_FOLDER/"data/params"
ds = load_dataset(path)
ds

ParAMSData(len=34, properties=['energy', 'forces'], transforms=[] at 0x7ebb7696cc20)

`out_properties` provide you an hint about the properties that you **MIGHT** find in the dataset entries.

In [3]:
ds.out_properties

[PropertyInfo(name='energy', unit='Ha', shape='float', description="Units might not be uniform! Found in ['training_set', 'validation_set']"),
 PropertyInfo(name='forces', unit='Ha/bohr', shape=(-1, 3), description="Units might not be uniform! Found in ['training_set', 'validation_set']")]

## Access the entries
The dataset supports `__getitem__` and `__iter__`, so you can retrieve
a single entry by index or loop through it.

In [4]:
from scm.moliterate import ChemDataEntry

# read one entry
entry: ChemDataEntry = ds[0]
entry

ChemDataEntry(COPt18, prop:['energy', 'forces'], md:['Frame', 'Origin', 'OriginalEnergyHartree', 'reference_engine', 'dataset'], idx_absolute=0, idx_origin=cleaned.job_frame001)

Each entry is a `ChemDataEntry`. The three main attributes you will use
most often are:
- `chemical_system` to obtain the `ChemicalSystem` class
- `properties` for computed values
- `metadata` for provenance and run details

In [5]:
entry.chemical_system

ChemicalSystem(charge: 0.0, formula: COPt18, 3D-periodic)

In [6]:
entry.properties

{'energy': -823.1489439350287,
 'forces': array([[ 5.61223336e-05,  3.98221179e-05,  4.10494034e-05],
        [-1.79948688e-05, -1.91575945e-05, -3.94670461e-05],
        [ 6.83522210e-03, -3.89991984e-03,  2.63242840e-02],
        [-5.01019647e-03,  2.86435305e-03,  1.39489783e-02],
        [-2.11524013e-04, -1.26684267e-04,  1.45319326e-02],
        [-5.60599594e-06,  2.96235962e-04,  1.45346165e-02],
        [ 2.51261912e-05,  7.83574826e-03,  2.62984853e-02],
        [ 2.42179146e-04, -1.37341495e-04,  1.45055899e-02],
        [-6.84006900e-03, -3.93367616e-03,  2.63249974e-02],
        [ 4.97399620e-03,  2.86269295e-03,  1.39259669e-02],
        [-1.91170016e-05, -5.79514979e-03,  1.39405364e-02],
        [ 7.62320020e-04, -2.29540888e-04, -1.85812636e-02],
        [ 5.70425653e-04, -5.24644096e-04, -1.85750107e-02],
        [ 1.99037924e-04,  7.89070542e-04, -1.85918705e-02],
        [ 3.80067063e-05, -3.51206472e-05, -1.58442090e-02],
        [-5.77440781e-04, -5.61916181e-04, -

In [7]:
entry.metadata

{'Frame': 1,
 'Origin': 'cleaned.job/ams.rkf',
 'OriginalEnergyHartree': -823.1489439350287,
 'reference_engine': 'None',
 'dataset': 'training_set'}

In [8]:
# iterate
for entry_i in ds:
    if entry_i.properties["energy"] > -823.13:
        print("high")

high
high
high
high
high
high
high
high
high


## Filter, slice
You can filter down to a subset and slice datasets like a list.

In [9]:
# slice
len(ds[:21])

21

In [10]:
from scm.moliterate.filters import RandomFilter

# filter
filter_call = RandomFilter(num_samples=3)
dss_subset = filter_call(ds)
len(dss_subset)

3

## Interfaces: ParAMS data, RKF, ASE

In [11]:
load_dataset(TUTORIAL_FOLDER/"data/params")

ParAMSData(len=34, properties=['energy', 'forces'], transforms=[] at 0x7ebb75575810)

In [12]:
load_dataset(TUTORIAL_FOLDER/"data/molecule.rkf")

RKFMolData(len=11, properties=['History%Gradients', 'History%maxGrad', 'History%maxStep', 'History%Energy', 'History%rmsStep', 'History%rmsGrad'], transforms=[] at 0x7ebb11584c20)

In [13]:
load_dataset(TUTORIAL_FOLDER/"data/periodic.rkf")

RKFMolData(len=20, properties=['History%Gradients', 'History%maxGrad', 'History%maxStep', 'History%Energy', 'History%rmsStep', 'History%rmsGrad'], transforms=[] at 0x7ebb180492c0)

The ase interface has also the option to be a writer. Therefore we can `create_dataset` and `add_systems`. For example here we import the data from the ParAMS format to the ase one. 

Note: the ase interface does not support calculator results storage, and only reads those properties.

In [14]:
from scm.moliterate import create_dataset

db = create_dataset(data_source=TUTORIAL_FOLDER/"ase.db", available_properties=ds.out_properties)
db.add_systems(ds)

100%|████████████████████████████████████████████████████████████████████████████████████| 34/34 [00:00<00:00, 752.07it/s]


In [15]:
db

ASEMolData(len=34, properties=['energy', 'forces'], transforms=[] at 0x7ebb115846d0)

In [16]:
db.unlink()

## Concatenate datasets

It is possible to work on concatenate datasets. 

Note: properties might be not uniform in this case!

In [17]:
path1 = TUTORIAL_FOLDER/"data/params"
path2 = TUTORIAL_FOLDER/"data/molecule.rkf"
path3 = TUTORIAL_FOLDER/"data/periodic.rkf"
dss = load_dataset([path1, path2, path3])
dss

ConcatChemDataSet(len=65, properties=['forces', 'History%Energy', 'History%maxGrad', 'History%maxStep', 'History%rmsStep', 'History%rmsGrad', 'History%Gradients', 'energy'], transforms=[] at 0x7ebb10ea9f90)

In [18]:
dss[0].properties.get("energy")

-823.1489439350287

In [19]:
dss[50].properties.get("energy", "energy not found!")

'energy not found!'

### Make uniform outputs with Transforms

To make uniform the output properties we could use the transforms.

In [20]:
from scm.moliterate.transforms import ConvertNamesTransform

from_rkf_to_params = ConvertNamesTransform(prop_names_convert={"History%Energy":"energy"})
dss.transforms.append(from_rkf_to_params)
dss

ConcatChemDataSet(len=65, properties=['forces', 'History%Energy', 'History%maxGrad', 'History%maxStep', 'History%rmsStep', 'History%rmsGrad', 'History%Gradients', 'energy'], transforms=[<class 'scm.moliterate.transforms.property_info_transform.ConvertNamesTransform'>] at 0x7ebb10ea9f90)

In [21]:
dss[0].properties.get("energy")

-823.1489439350287

In [22]:
dss[50].properties.get("energy", "energy not found!")

-1099.7486339573088

# Next Steps

Other tutorials in increasing order of complexity:

- __[core-moliterate-concepts.ipynb](core-moliterate-concepts.ipynb)__
- __[advanced-features-filters.ipynb](advanced-features-filters.ipynb)__
- __[advanced-create-an-interface.ipynb](advanced-create-an-interface.ipynb)__
